# Data Validation

In [1]:
import xarray as xr
import numpy as np

In [2]:
ds = xr.open_dataset('/Users/jacobalbertogarcia/portfolio-projects/projects/02_wind_extremes_ph/data/raw/era5_u10v10_2020_PH.nc')

In [3]:
ds

<xarray.Dataset> Size: 279MB
Dimensions:     (valid_time: 8784, latitude: 81, longitude: 49)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 70kB 2020-01-01 ... 2020-12-31T23...
    expver      (valid_time) <U4 141kB ...
  * latitude    (latitude) float64 648B 22.0 21.75 21.5 21.25 ... 2.5 2.25 2.0
  * longitude   (longitude) float64 392B 116.0 116.2 116.5 ... 127.5 127.8 128.0
    number      int64 8B ...
Data variables:
    u10         (valid_time, latitude, longitude) float32 139MB ...
    v10         (valid_time, latitude, longitude) float32 139MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-02-17T01:23 GRIB to CDM+CF via cfgrib-0.9.1...

## Confirm coordinates and time

In [4]:
print(ds.latitude.min().item(), ds.latitude.max().item())
print(ds.longitude.min().item(), ds.longitude.max().item())
print(ds.valid_time.values[0], ds.valid_time.values[-1])

2.0 22.0
116.0 128.0
2020-01-01T00:00:00.000000000 2020-12-31T23:00:00.000000000


## Compute wind speed

In [5]:
ws10 = np.sqrt(ds['u10']**2 + ds['v10']**2)
ws10

<xarray.DataArray (valid_time: 8784, latitude: 81, longitude: 49)> Size: 139MB
array([[[12.478109  , 12.534944  , 12.761858  , ..., 12.255883  ,
         12.48994   , 12.658661  ],
        [12.560815  , 12.609189  , 12.743814  , ..., 12.709398  ,
         12.856747  , 12.904666  ],
        [12.638994  , 12.811718  , 12.9304695 , ..., 12.756969  ,
         12.8501835 , 12.917098  ],
        ...,
        [ 1.3540263 ,  1.4917156 ,  0.91439265, ...,  6.7701354 ,
          6.577974  ,  5.7302103 ],
        [ 1.3886881 ,  1.5467533 ,  1.2791362 , ...,  6.3077407 ,
          5.6796265 ,  4.785828  ],
        [ 1.1403335 ,  1.7156366 ,  1.5450684 , ...,  5.3420596 ,
          4.7453947 ,  4.9085054 ]],

       [[12.3057    , 12.368518  , 12.564924  , ..., 12.116138  ,
         12.326628  , 12.451254  ],
        [12.339623  , 12.400762  , 12.556161  , ..., 12.58406   ,
         12.753604  , 12.83731   ],
        [12.414115  , 12.673833  , 12.823924  , ..., 12.944843  ,
         13.041539  , 13.140403  ],
...
        [ 1.1580728 ,  1.3901734 ,  1.3432829 , ...,  1.1638306 ,
          0.6411357 ,  0.15907276],
        [ 0.7529365 ,  1.0033151 ,  1.4296135 , ...,  0.7327241 ,
          0.42403394,  0.3727969 ],
        [ 0.44442424,  0.840095  ,  1.3926234 , ...,  0.31826857,
          0.16294344,  0.67408377]],

       [[11.062168  , 11.33195   , 11.796174  , ...,  8.709441  ,
          8.437691  ,  8.387486  ],
        [11.509862  , 11.737537  , 11.93521   , ...,  8.65427   ,
          8.581094  ,  8.661178  ],
        [11.650244  , 11.871074  , 11.972436  , ...,  8.923353  ,
          8.928782  ,  9.112828  ],
        ...,
        [ 1.1590923 ,  1.0881304 ,  0.97637635, ...,  0.7187801 ,
          0.3671141 ,  0.4196076 ],
        [ 1.0668107 ,  1.0192549 ,  1.1140686 , ...,  0.5060488 ,
          0.40934086,  0.7973082 ],
        [ 0.785345  ,  1.1584712 ,  1.3001692 , ...,  0.40426537,
          0.6461065 ,  1.0273429 ]]], shape=(8784, 81, 49), dtype=float32)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 70kB 2020-01-01 ... 2020-12-31T23...
    expver      (valid_time) <U4 141kB '0001' '0001' '0001' ... '0001' '0001'
  * latitude    (latitude) float64 648B 22.0 21.75 21.5 21.25 ... 2.5 2.25 2.0
  * longitude   (longitude) float64 392B 116.0 116.2 116.5 ... 127.5 127.8 128.0
    number      int64 8B 0
Attributes: (12/27)
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      3969
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    GRIB_gridType:                            regular_ll
    ...                                       ...
    GRIB_missingValue:                        3.4028234663852886e+38
    GRIB_totalNumber:                         0
    GRIB_units:                               m s**-1
    units:                                    m s**-1
    standard_name:                            unknown
    GRIB_surface:                             0.0

## Add to DataSet object

In [6]:
ds = ds.assign(ws10=ws10)
ds['ws10'].attrs['units'] = 'm s-1'
ds["ws10"].attrs["long_name"] = "10m wind speed"

In [7]:
ds

<xarray.Dataset> Size: 419MB
Dimensions:     (valid_time: 8784, latitude: 81, longitude: 49)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 70kB 2020-01-01 ... 2020-12-31T23...
    expver      (valid_time) <U4 141kB ...
  * latitude    (latitude) float64 648B 22.0 21.75 21.5 21.25 ... 2.5 2.25 2.0
  * longitude   (longitude) float64 392B 116.0 116.2 116.5 ... 127.5 127.8 128.0
    number      int64 8B ...
Data variables:
    u10         (valid_time, latitude, longitude) float32 139MB -9.738 ... 1.024
    v10         (valid_time, latitude, longitude) float32 139MB -7.802 ... 0....
    ws10        (valid_time, latitude, longitude) float32 139MB 12.48 ... 1.027
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-02-17T01:23 GRIB to CDM+CF via cfgrib-0.9.1...